<a href="https://colab.research.google.com/github/Ming-flowbetter/Machine-Learning/blob/main/lec01_contamination_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 01 demo: detecting benchmark contamination

CSCI 567 --- Machine Learning

No model is trained in this notebook. Instead we build, from scratch, a version of the
**n-gram overlap** contamination check that Brown et al. (2020, the GPT-3 paper, Appendix C)
used to measure whether benchmark test examples had leaked into a model's pretraining data.
Their method used 13-gram overlap at internet scale; we use a smaller `n` on toy text so the
mechanics are visible by eye.

By the end you will have:
1. A toy "pretraining corpus" and a toy "benchmark" with some examples that leaked in
   (verbatim or near-duplicate) and some that did not.
2. An n-gram overlap detector that flags the leaked ones.
3. A look at why near-duplicates (not just exact copies) make this harder than it sounds.

In [ ]:
import re
from itertools import islice

def tokenize(text):
    """Lowercase, strip punctuation, split on whitespace."""
    text = re.sub(r"[^a-z0-9\s]", "", text.lower())
    return text.split()

def ngrams(tokens, n):
    """Set of n-grams (as tuples) in a token list."""
    return {tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)}

## 1. Build a toy pretraining corpus and a toy benchmark

Three benchmark examples: one leaked verbatim into the pretraining corpus, one leaked as a
near-duplicate (a paraphrase that still shares a long run of exact tokens), and one that is
genuinely clean.

In [ ]:
pretraining_corpus = [
    "The capital of France is Paris, a city known for the Eiffel Tower and the Louvre museum.",
    "In machine learning, a model that memorizes training data fails to generalize to new data.",
    "Photosynthesis is the process by which plants convert sunlight into chemical energy stored as glucose.",
]

benchmark = {
    "leaked_verbatim": "The capital of France is Paris, a city known for the Eiffel Tower and the Louvre museum.",
    "leaked_near_duplicate": "In machine learning, a model that just memorizes its training data fails to generalize to new data.",
    "clean": "The mitochondria is the organelle responsible for producing most of a cell's ATP through respiration.",
}

## 2. The GPT-3-style n-gram overlap check

Flag a benchmark example as contaminated if it shares an n-gram of length `n` with anything in
the pretraining corpus. GPT-3 used `n = 13`; we use a much smaller `n` since our toy corpus is
only a few sentences long.

In [ ]:
def corpus_ngrams(corpus, n):
    all_ngrams = set()
    for doc in corpus:
        all_ngrams |= ngrams(tokenize(doc), n)
    return all_ngrams

def contamination_overlap(example, corpus, n):
    """Return the set of n-grams the example shares with the pretraining corpus."""
    example_grams = ngrams(tokenize(example), n)
    return example_grams & corpus_ngrams(corpus, n)

N = 6  # toy-scale analog of GPT-3's n=13

for name, text in benchmark.items():
    overlap = contamination_overlap(text, pretraining_corpus, N)
    verdict = "CONTAMINATED" if overlap else "clean"
    print(f"{name:24s} [{verdict:12s}]  shared {N}-grams: {len(overlap)}")
    if overlap:
        print(f"  e.g. {next(iter(overlap))}")

leaked_verbatim          [CONTAMINATED]  shared 6-grams: 12
  e.g. ('is', 'paris', 'a', 'city', 'known', 'for')
leaked_near_duplicate    [CONTAMINATED]  shared 6-grams: 4
  e.g. ('in', 'machine', 'learning', 'a', 'model', 'that')
clean                    [clean       ]  shared 6-grams: 0


Run the cell above. You should see:
- `leaked_verbatim` flagged immediately (every n-gram matches).
- `leaked_near_duplicate` flagged too, *if* the paraphrase still preserves an unbroken run of
  `N` tokens somewhere (try lowering/raising `N` and see when this stops catching it).
- `clean` never flagged, correctly.

**Exercise.** Rewrite `leaked_near_duplicate` so it means the same thing but shares no run of
`N` consecutive tokens with the corpus (e.g. reorder clauses, use synonyms throughout). Confirm
the detector now misses it. This is the core limitation Sainz et al. (2023) raise: n-gram
overlap only catches *lexical* leakage, not paraphrased or translated leakage, which is why
they argue the true extent of contamination is likely underestimated by this method alone.

## 3. A second method: guided-instruction completion (Golchin & Surdeanu, 2024)

Instead of comparing text you already have, this method *probes* a model: give it the dataset
name, the partition, and a truncated fragment of a real benchmark instance, and see if it
completes the rest suspiciously well. We won't query a real pretrained LLM here, but we can
simulate the *comparison* step: given a candidate completion, how would you decide if it's
"suspiciously close" to the true continuation versus a plausible guess?

In [ ]:
def completion_overlap_score(true_continuation, candidate_completion, n=3):
    """A simple stand-in for a contamination score: n-gram overlap ratio.
    A real study would use this alongside a plausibility baseline from an
    uncontaminated model -- a high raw score alone isn't proof, only a signal.
    """
    true_grams = ngrams(tokenize(true_continuation), n)
    cand_grams = ngrams(tokenize(candidate_completion), n)
    if not true_grams:
        return 0.0
    return len(true_grams & cand_grams) / len(true_grams)

true_continuation = "a city known for the Eiffel Tower and the Louvre museum."
model_a_completion = "a city known for the Eiffel Tower and the Louvre museum."  # suspiciously exact
model_b_completion = "one of the most visited cities in the world."             # plausible guess

for name, completion in [("model_a (suspect)", model_a_completion), ("model_b (plausible)", model_b_completion)]:
    score = completion_overlap_score(true_continuation, completion)
    print(f"{name:22s} overlap score: {score:.2f}")

model_a (suspect)      overlap score: 1.00
model_b (plausible)    overlap score: 0.00


## Discussion

- Brown et al. (2020) used 13-gram overlap at web scale and found only a negligible effect on
  GPT-3's benchmark scores in most cases -- but flagged a bug in their own filtering that
  undercounted overlaps.
- Sainz et al. (2023) argue the field-wide extent of contamination is largely unmeasured, not
  shown to be small, and that lexical n-gram methods like the one above likely miss paraphrased
  leakage.
- Golchin & Surdeanu (2024) probe the *model* directly rather than comparing static text, which
  catches leakage the n-gram method above cannot (the model may have seen a paraphrase, not the
  exact string).

Full citations: `references/sources/lecture01-generalization-evaluation-citations.md`.